# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: detect local pack folder or current working directory
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif (Path.cwd() / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats"
elif (Path.cwd().parent / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent / "flasheats"
elif (Path.cwd() / "flasheats-classroom-pack" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats-classroom-pack"
elif (Path.cwd() / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats"
elif (Path.cwd().parent / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent / "flasheats"
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()
elif (Path.cwd() / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats"
elif (Path.cwd().parent / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent / "flasheats"

if BASE is None:
    BASE = find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded = files.upload()
        zip_name = next(n for n in uploaded if n.endswith(".zip"))
        extract_dir = Path("/content/flasheats_class7")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE = find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE = find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats database/pack")
print("Using pack:", BASE)


Using pack: flasheats


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders dynamically:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Multi-Source Timeline Reconstruction
We integrate lifecycle milestones across all operational sources:
1. `order_events.csv` (platform core states: `ORDER_CREATED`, `PICKED_UP`, `DELIVERED`)
2. `customer_app_actions.csv` (customer mobile telemetry: `ETA_VIEWED`, `CANCEL_ATTEMPTED`, `SUPPORT_CONTACTED`)
3. `order_interventions.csv` (operations console actions: `PRIORITY_DISPATCH`, `DRIVER_REASSIGNMENT`, `RESTAURANT_CONTACT`, `CUSTOMER_CREDIT`)
4. `support_tickets.csv` (customer support CRM complaints)
5. `restaurant_status.csv` (merchant kitchen display updates: `PREPARING`, `READY`, `HANDED_OFF`)

In [3]:
# Challenge 1 Code: Reconstructing Order Lifecycle Timelines

base_orders = orders.drop_duplicates("order_id", keep="first")
late_order_id = outcomes[outcomes.late_flag == 1.0].order_id.iloc[0]
on_time_order_id = outcomes[outcomes.late_flag == 0.0].order_id.iloc[0]
intervention_order_id = interventions.order_id.iloc[0]

print(f"Selected Orders -> On-Time: {on_time_order_id}, Late: {late_order_id}, Intervened: {intervention_order_id}")

# Load supplementary order_events
order_events = pd.read_csv(BASE / "data" / "order_events.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

def build_order_timeline(target_order_id):
    timeline = []
    
    # 1. Platform order events
    ev_sub = order_events[order_events['order_id'] == target_order_id]
    for _, r in ev_sub.iterrows():
        timeline.append({
            'event_time': str(r['event_time']),
            'event_type': r['event_type'],
            'actor': r['actor_type'],
            'source_system': r['source_system']
        })
        
    # 2. Customer app actions
    act_sub = customer_actions[customer_actions['order_id'] == target_order_id]
    for _, r in act_sub.iterrows():
        timeline.append({
            'event_time': str(r['action_at']),
            'event_type': f"APP_{r['action_type']}",
            'actor': 'customer',
            'source_system': f"customer_app_{r['channel']}"
        })
        
    # 3. Interventions
    int_sub = interventions[interventions['order_id'] == target_order_id]
    for _, r in int_sub.iterrows():
        timeline.append({
            'event_time': str(r['intervention_at']),
            'event_type': f"OPS_INTERVENTION_{r['intervention_type']}",
            'actor': r['initiated_by'],
            'source_system': 'operations_console'
        })
        
    # 4. Support tickets
    tkt_sub = tickets[tickets['order_id'] == target_order_id]
    for _, r in tkt_sub.iterrows():
        timeline.append({
            'event_time': str(r['created_at']),
            'event_type': f"TICKET_OPENED_{r['category']}",
            'actor': 'customer',
            'source_system': 'support_desk'
        })
        
    # 5. Restaurant status
    rs_sub = restaurant_status[restaurant_status['order_id'] == target_order_id]
    for _, r in rs_sub.iterrows():
        timeline.append({
            'event_time': str(r['last_updated_at']),
            'event_type': f"KITCHEN_{r['status'].strip().upper() if isinstance(r['status'], str) else 'STATUS'}",
            'actor': 'restaurant',
            'source_system': 'merchant_app'
        })
        
    df_timeline = pd.DataFrame(timeline)
    if not df_timeline.empty:
        df_timeline = df_timeline.sort_values('event_time').reset_index(drop=True)
    return df_timeline

for label, oid in [("Delivered On-Time", on_time_order_id), ("Delivered Late", late_order_id), ("With Operational Intervention", intervention_order_id)]:
    print("=" * 75)
    print(f"ORDER LIFECYCLE TIMELINE: {label} (Order ID: {oid})")
    print("=" * 75)
    display(build_order_timeline(oid))


Selected Orders -> On-Time: O00003, Late: O00001, Intervened: O00781
ORDER LIFECYCLE TIMELINE: Delivered On-Time (Order ID: O00003)
                   event_time         event_type       actor   source_system
0         2026-08-01T19:35:00      ORDER_CREATED    customer  order_platform
1         2026-08-01T19:48:00  KITCHEN_PREPARING  restaurant    merchant_app
2  2026-08-01T19:57:50.820366          PICKED_UP      driver  order_platform
3  2026-08-01T20:13:28.061191          DELIVERED      driver  order_platform
ORDER LIFECYCLE TIMELINE: Delivered Late (Order ID: O00001)
                   event_time                            event_type     actor            source_system
0         2026-08-13T12:56:00                         ORDER_CREATED  customer           order_platform
1         2026-08-13T13:08:00  OPS_INTERVENTION_DRIVER_REASSIGNMENT  dispatch       operations_console
2         2026-08-13T13:21:00                        APP_ETA_VIEWED  customer  customer_app_mobile_app
3  2026-08-

# Challenge 2 — Define the canonical project model

Your model must support:
1. **customer → orders**
2. **order → customer interactions**
3. **order → support interactions**
4. **order → interventions**
5. **order → outcome**

### Entity-Relationship Architecture (Workflow-Centric)

| Entity / Table | Primary Key | Foreign Keys | Business Grain | Description |
|---|---|---|---|---|
| `customers` | `customer_id` | - | 1 row per customer | Customer master profile and location. |
| `orders` | `order_id` | `customer_id`, `restaurant_id`, `driver_id` | 1 row per order | Core transactional record containing committed ETA, timestamps, and routing attributes. |
| `customer_app_actions` | `action_id` | `order_id`, `customer_id` | 1 row per customer mobile action | Pre-delivery customer interaction events (`ETA_VIEWED`, `CANCEL_ATTEMPTED`). |
| `support_tickets` | `ticket_id` | `order_id` | 1 row per complaint ticket | Post-order or in-flight customer escalation and complaint category. |
| `order_interventions` | `intervention_id` | `order_id` | 1 row per operational action | Corrective operational adjustments made by dispatch/support (`PRIORITY_DISPATCH`, etc.). |
| `order_outcomes` | `order_id` | `order_id` | 1 row per completed order | Evaluated SLA outcome (`delivered_flag`, `late_flag`, `delay_min`, `outcome_bucket`). |

### Why this workflow model is superior to mirroring source-system tables:
1. **Eliminates Cartesian Join Explosions**: Raw source tables represent 1-to-many event streams (an order can have 8 app actions, 2 support tickets, and 3 interventions). Joining raw tables directly duplicates rows and corrupts financial/volume metrics.
2. **Maps to the Customer & Operational Journey**: Instead of isolating CRM tickets from dispatch logs, the workflow model links cause and effect: `Order Placement → Friction / Escalation → Operational Remedy → Final Outcome`.
3. **Focuses on Project KPI**: Isolates the critical decision levers (`interventions`) and customer pain points (`actions/tickets`) directly against the outcome metric (`delay_min` / `late_flag`).

In [4]:
# Challenge 2: Document Source Grains and Foreign Key Integrity

sources = {
    "orders": orders,
    "customer_actions": customer_actions,
    "support_tickets": tickets,
    "interventions": interventions,
    "outcomes": outcomes
}

print("=" * 60)
print("CANONICAL SOURCE GRAINS & RELATIONSHIPS:")
for name, df in sources.items():
    pk_candidates = [c for c in df.columns if c.endswith('_id') and df[c].nunique() == len(df)]
    pk = pk_candidates[0] if pk_candidates else "(compound/none)"
    fk_orders = "order_id" in df.columns
    order_nunique = df["order_id"].nunique() if fk_orders else "N/A"
    print(f"- {name:20s}: Shape={str(df.shape):12s} | PK={pk:15s} | Unique Orders={order_nunique}")
print("=" * 60)


CANONICAL SOURCE GRAINS & RELATIONSHIPS:
- orders              : Shape=(1603, 13)   | PK=(compound/none) | Unique Orders=1600
- customer_actions    : Shape=(2365, 6)    | PK=action_id       | Unique Orders=1100
- support_tickets     : Shape=(202, 5)     | PK=(compound/none) | Unique Orders=198
- interventions       : Shape=(430, 6)     | PK=intervention_id | Unique Orders=430
- outcomes            : Shape=(1600, 6)    | PK=order_id        | Unique Orders=1600


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

### FDE Pre-Aggregation Strategy
To avoid row multiplication, we pre-aggregate every one-to-many dataset to `order_id` grain before performing left joins:
- `customer_app_actions` → aggregated to `cancel_attempted` flag and app action counts.
- `support_tickets` → aggregated to `ticket_count` and ticket-opened flag.
- Combined `support_opened` flag = 1 if the customer opened a support ticket or contacted support in-app.
- `order_interventions` → aggregated to `intervention_count` and distinct `intervention_types` string.
- Target table grain: **exactly 1 row per order (1600 rows, 0 duplicate order_ids)**.

In [5]:
# Challenge 3: Pre-aggregate 1-to-many tables and build canonical order-level dataset

# 1. Aggregate customer app actions to order level
actions_by_order = customer_actions.groupby("order_id").agg(
    app_action_count=("action_id", "count"),
    cancel_attempted=("action_type", lambda s: int((s == "CANCEL_ATTEMPTED").any())),
    support_opened_app=("action_type", lambda s: int((s.isin(["SUPPORT_CONTACTED", "HELP_CLICKED"])).any()))
).reset_index()

# 2. Aggregate support tickets to order level
valid_tickets = tickets[tickets["order_id"].notna()].copy()
tickets_by_order = valid_tickets.groupby("order_id").agg(
    support_ticket_count=("ticket_id", "count")
).reset_index()
tickets_by_order["support_opened_ticket"] = 1

# 3. Aggregate interventions to order level
interventions_by_order = interventions.groupby("order_id").agg(
    intervention_count=("intervention_id", "count"),
    intervention_types=("intervention_type", lambda s: "; ".join(sorted(s.unique())))
).reset_index()

# 4. Join onto deduplicated orders and outcomes
base_orders = orders.drop_duplicates("order_id", keep="first")[["order_id", "customer_id", "restaurant_id", "driver_id"]].copy()

order_level_df = outcomes.merge(base_orders, on="order_id", how="left")
order_level_df = order_level_df.merge(actions_by_order, on="order_id", how="left")
order_level_df = order_level_df.merge(tickets_by_order, on="order_id", how="left")
order_level_df = order_level_df.merge(interventions_by_order, on="order_id", how="left")

# Clean flags
order_level_df["cancel_attempted"] = order_level_df["cancel_attempted"].fillna(0).astype(int)
order_level_df["support_opened_app"] = order_level_df["support_opened_app"].fillna(0).astype(int)
order_level_df["support_opened_ticket"] = order_level_df["support_opened_ticket"].fillna(0).astype(int)
order_level_df["support_opened"] = ((order_level_df["support_opened_app"] == 1) | (order_level_df["support_opened_ticket"] == 1)).astype(int)

order_level_df["intervention_count"] = order_level_df["intervention_count"].fillna(0).astype(int)
order_level_df["intervention_types"] = order_level_df["intervention_types"].fillna("NONE")
order_level_df["final_status"] = order_level_df["final_status_norm"]

# Select the exact requested columns
canonical_columns = [
    "order_id", "customer_id", "support_opened", "cancel_attempted",
    "intervention_count", "intervention_types", "final_status", "late_flag", "delay_min"
]
order_model = order_level_df[canonical_columns].copy()

# Programmatic verification of grain
total_model_rows = len(order_model)
unique_model_orders = order_model["order_id"].nunique()
model_duplicates = order_model["order_id"].duplicated().sum()

print("=" * 60)
print("ORDER-LEVEL CANONICAL MODEL VERIFICATION:")
print(f"- Total Rows:       {total_model_rows}")
print(f"- Unique Orders:    {unique_model_orders}")
print(f"- Duplicate Rows:   {model_duplicates}")
print(f"- Verified 1 row = 1 order: {total_model_rows == unique_model_orders and model_duplicates == 0}")
print("=" * 60)
display(order_model.head(10))

# Compute answers to the 4 business questions dynamically
late_orders_mask = order_model["late_flag"] == 1.0
total_late_orders = late_orders_mask.sum()
late_with_support = (late_orders_mask & (order_model["support_opened"] == 1)).sum()

total_intervened_orders = (order_model["intervention_count"] > 0).sum()

common_intervention = interventions["intervention_type"].value_counts()

frustrated_no_int = order_model[
    ((order_model["support_opened"] == 1) | (order_model["cancel_attempted"] == 1)) &
    (order_model["intervention_count"] == 0)
]

print("\n" + "=" * 60)
print("CHALLENGE 3: BUSINESS QUESTION ANSWERS (DERIVED DYNAMICALLY)")
print("=" * 60)
print(f"1. How many late orders had support interaction?")
print(f"   Answer: {late_with_support} out of {total_late_orders} late orders ({late_with_support/total_late_orders*100:.2f}%).")
print(f"   WHAT THE DATA SHOWS: ~19% of late customers actively opened support.")
print(f"   WHAT THE DATA DOES NOT PROVE: Does NOT mean remaining 81% were satisfied; silent customer churn is unobserved.")

print(f"\n2. How many orders received intervention?")
print(f"   Answer: {total_intervened_orders} out of {total_model_rows} orders ({total_intervened_orders/total_model_rows*100:.2f}%).")
print(f"   WHAT THE DATA SHOWS: Over a quarter of all orders trigger at least one operational intervention.")
print(f"   WHAT THE DATA DOES NOT PROVE: Interventions alone do not guarantee recovery.")

print(f"\n3. Which intervention is most common?")
print(f"   Answer: '{common_intervention.index[0]}' with {common_intervention.iloc[0]} occurrences.")
print("   Full Breakdown:")
display(common_intervention)

print(f"\n4. Which frustrated journeys had no intervention?")
print(f"   Answer: {len(frustrated_no_int)} orders had customer friction (support contacted or cancel attempted) but received ZERO interventions.")
print("   Sample unassisted frustrated journeys:")
display(frustrated_no_int.head(5))
print("=" * 60)


ORDER-LEVEL CANONICAL MODEL VERIFICATION:
- Total Rows:       1600
- Unique Orders:    1600
- Duplicate Rows:   0
- Verified 1 row = 1 order: True
  order_id customer_id  support_opened  cancel_attempted  intervention_count   intervention_types final_status  late_flag  delay_min
0   O00001       C0168               0                 0                   1  DRIVER_REASSIGNMENT    delivered        1.0      10.82
1   O00002       C0043               0                 0                   0                 NONE    delivered        1.0       3.97
2   O00003       C0855               0                 0                   0                 NONE    delivered        0.0      -4.41
3   O00004       C0229               0                 0                   0                 NONE    cancelled        NaN        NaN
4   O00005       C0040               0                 0                   0                 NONE    delivered        1.0      21.02
5   O00006       C0152               0                 

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

### Selected Operational Metrics

| Metric Name | Formula | Grain | Category | Why It Matters | Linkage to Project KPI |
|---|---|---|---|---|---|
| **1. Late Delivery Rate** | `Count(orders with delay_min > 0) / Count(delivered orders with non-null actual_delivery_at)` | Order Outcome | **Outcome** | Primary SLA baseline measuring commitment fulfillment. | Direct Project KPI. Tracking reduction over time measures overall platform success. |
| **2. Customer Friction Rate** | `Count(orders with support_opened == 1 or cancel_attempted == 1) / Count(total orders)` | Order Customer Interaction | **Interaction** | Leading indicator of churn; identifies orders where delay degraded customer trust. | Directly driven by late deliveries; reducing lateness prevents customer friction and support costs. |
| **3. Operational Intervention Rate** | `Count(orders with intervention_count > 0) / Count(total orders)` | Order Operational Workflow | **Intervention** | Measures operational overhead and how often orders deviate from automated flow. | High intervention rate indicates dispatch/kitchen instability; operational fixes aim to reduce exceptions. |
| **4. Intervention Rescue Rate** | `Count(intervened orders delivered on time) / Count(intervened orders)` | Order Operational Performance | **Intervention Recovery** | Evaluates whether operations successfully prevents late delivery when stepping in. | Directly impacts late delivery rate by converting at-risk orders into on-time completions. |

In [6]:
# Challenge 4: Compute Selected Operational Metrics Programmatically

valid_outcomes = outcomes[outcomes.late_flag.notna()]
total_delivered_evaluated = len(valid_outcomes)
total_late = (valid_outcomes.late_flag == 1.0).sum()
late_delivery_rate = (total_late / total_delivered_evaluated) * 100.0

friction_orders = ((order_model["support_opened"] == 1) | (order_model["cancel_attempted"] == 1)).sum()
friction_rate = (friction_orders / len(order_model)) * 100.0

intervened_total = (order_model["intervention_count"] > 0).sum()
intervention_rate = (intervened_total / len(order_model)) * 100.0

intervened_evaluated = order_model[(order_model["intervention_count"] > 0) & (order_model["late_flag"].notna())]
rescued_orders = (intervened_evaluated["late_flag"] == 0.0).sum()
rescue_rate = (rescued_orders / len(intervened_evaluated)) * 100.0

metrics_table = pd.DataFrame([
    {"Metric Name": "1. Late Delivery Rate", "Type": "Outcome", "Numerator": total_late, "Denominator": total_delivered_evaluated, "Calculated Value": f"{late_delivery_rate:.2f}%"},
    {"Metric Name": "2. Customer Friction Rate", "Type": "Interaction", "Numerator": friction_orders, "Denominator": len(order_model), "Calculated Value": f"{friction_rate:.2f}%"},
    {"Metric Name": "3. Operational Intervention Rate", "Type": "Intervention", "Numerator": intervened_total, "Denominator": len(order_model), "Calculated Value": f"{intervention_rate:.2f}%"},
    {"Metric Name": "4. Intervention Rescue Rate", "Type": "Recovery", "Numerator": rescued_orders, "Denominator": len(intervened_evaluated), "Calculated Value": f"{rescue_rate:.2f}%"},
])

print("=" * 60)
print("CALCULATED PROJECT OPERATIONAL METRICS:")
display(metrics_table)
print("=" * 60)


CALCULATED PROJECT OPERATIONAL METRICS:
                        Metric Name          Type  Numerator  Denominator Calculated Value
0             1. Late Delivery Rate       Outcome        843         1495           56.39%
1         2. Customer Friction Rate   Interaction        201         1600           12.56%
2  3. Operational Intervention Rate  Intervention        430         1600           26.88%
3       4. Intervention Rescue Rate      Recovery        176          404           43.56%


# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three questions with the required FDE distinction:

> **What does this tell the business, and what does it NOT prove?**

We investigate:
- **A. Do orders with support interactions have higher delay?**
- **B. What is the late rate with vs without intervention?**
- **C. Which intervention type is associated with the lowest late rate?**
- **D. Which restaurants contribute the largest number of late orders?**
- **E. Which journeys show support interaction + intervention + still late?**

In [7]:
# Challenge 5: Investigate the Workflow with Joins and Aggregations

print("=" * 75)
print("A. DO ORDERS WITH SUPPORT INTERACTIONS HAVE HIGHER DELAY?")
delay_by_supp = order_model[order_model["delay_min"].notna()].groupby("support_opened")["delay_min"].agg(["count", "mean", "median", lambda x: x.quantile(0.90)])
delay_by_supp.columns = ["Orders", "Mean Delay (min)", "Median Delay (min)", "90th Pct Delay"]
display(delay_by_supp)
print("WHAT THE DATA SHOWS: Orders with support interactions average 10.99 min delay vs 1.93 min for orders without support (median 11.43m vs 0.50m).")
print("WHAT THE DATA DOES NOT PROVE: Support does not cause delays. Customers reach out because an order is severely delayed (correlation, not causation).")

print("\n" + "=" * 75)
print("B. WHAT IS LATE RATE WITH VS WITHOUT INTERVENTION?")
late_by_int = order_model[order_model["late_flag"].notna()].groupby(order_model["intervention_count"] > 0)["late_flag"].agg(["count", "mean"])
late_by_int.columns = ["Orders", "Late Rate"]
late_by_int["Late Rate (%)"] = late_by_int["Late Rate"] * 100.0
display(late_by_int)
print("WHAT THE DATA SHOWS: The late rate is virtually identical: 56.44% with intervention vs 56.37% without intervention.")
print("WHAT THE DATA DOES NOT PROVE: This does NOT prove interventions are ineffective. Interventions suffer from adverse selection — operations only intervenes when an order is already at severe risk of being late.")

print("\n" + "=" * 75)
print("C. WHICH INTERVENTION TYPE IS ASSOCIATED WITH THE LOWEST LATE RATE?")
int_eval = interventions.merge(outcomes[["order_id", "late_flag", "delay_min"]], on="order_id", how="left")
int_type_perf = int_eval.groupby("intervention_type")["late_flag"].agg(["count", "mean"])
int_type_perf.columns = ["Interventions", "Late Rate"]
int_type_perf["Late Rate (%)"] = (int_type_perf["Late Rate"] * 100.0).round(2)
display(int_type_perf.sort_values("Late Rate (%)"))
print("WHAT THE DATA SHOWS: 'PRIORITY_DISPATCH' has the lowest late rate (51.16%), while 'CUSTOMER_CREDIT' has the highest (66.67%).")
print("WHAT THE DATA DOES NOT PROVE: Priority dispatch is applied mid-journey to accelerate transit, whereas customer credit is applied post-facto as compensation after late delivery occurred.")

print("\n" + "=" * 75)
print("D. WHICH RESTAURANTS CONTRIBUTE THE LARGEST NUMBER OF LATE ORDERS?")
rest_late_counts = outcomes[outcomes["late_flag"] == 1.0].merge(orders[["order_id", "restaurant_id"]].drop_duplicates("order_id"), on="order_id")
top_late_rest = rest_late_counts["restaurant_id"].value_counts().head(5).reset_index()
top_late_rest.columns = ["Restaurant ID", "Late Orders Count"]
display(top_late_rest)
print("WHAT THE DATA SHOWS: Top 5 restaurants (R050, R004, R051, R024, R023) generate 102 late orders (~12% of all late orders).")
print("WHAT THE DATA DOES NOT PROVE: Does not prove kitchen incompetence; high late order counts may be driven by high sales volume or long delivery distances.")

print("\n" + "=" * 75)
print("E. DOUBLE-FAILURE JOURNEYS (SUPPORT + INTERVENTION + STILL LATE):")
double_failures = order_model[(order_model["support_opened"] == 1) & (order_model["intervention_count"] > 0) & (order_model["late_flag"] == 1.0)]
print(f"Count of high-friction double-failure journeys: {len(double_failures)} orders")
display(double_failures[["order_id", "customer_id", "intervention_types", "delay_min"]].head(5))
print("=" * 75)


A. DO ORDERS WITH SUPPORT INTERACTIONS HAVE HIGHER DELAY?
                Orders  Mean Delay (min)  Median Delay (min)  90th Pct Delay
support_opened                                                              
0                 1298          1.933297               0.495           15.52
1                  197         10.987665              11.430           25.22
WHAT THE DATA SHOWS: Orders with support interactions average 10.99 min delay vs 1.93 min for orders without support (median 11.43m vs 0.50m).
WHAT THE DATA DOES NOT PROVE: Support does not cause delays. Customers reach out because an order is severely delayed (correlation, not causation).

B. WHAT IS LATE RATE WITH VS WITHOUT INTERVENTION?
                    Orders  Late Rate  Late Rate (%)
intervention_count                                  
False                 1091   0.563703      56.370302
True                   404   0.564356      56.435644
WHAT THE DATA SHOWS: The late rate is virtually identical: 56.44% with interven

# Challenge 6 — Connect the model to the KPI

### One-Page Project View: Connecting Data to the Project KPI

```
PROJECT KPI: Reduce Late Delivery Rate
   │
   ├── OUTCOME METRIC: Late Delivery Rate (% > 0 min) & P90 Delivery Delay
   │
   ├── WORKFLOW / DRIVER METRICS:
   │     ├── Kitchen Prep Dwell Time (Estimated)
   │     ├── Driver Assignment Latency
   │     └── Customer Friction Rate (App ETA Re-checks & Cancellations)
   │
   ├── INTERVENTIONS:
   │     ├── Proactive Priority Dispatch (Lowest late rate among remedies: 51.2%)
   │     ├── Driver Reassignment
   │     └── Merchant Contact
   │
   └── DATA SOURCES & EVENTS:
         ├── orders / order_outcomes (Promised ETA, Actual Delivery, Status)
         ├── driver_events (Assignment, Pickup, Delivery, GPS)
         ├── customer_app_actions (In-flight ETA views, cancellations)
         └── support_tickets (Customer escalations)
```

### Strategic Executive Questions

1. **Which metrics are directly controllable by operations?**
   - **Controllable**: Driver assignment speed, priority dispatch triggers, proactive notifications to customers before ETA expires, and automated driver re-routing.
   - **Uncontrollable (Exogenous)**: Severe weather, city-wide traffic jams, and customer phone response latency.

2. **Which are outcomes?**
   - Final delivery lateness (`late_flag`, `delay_min`), customer cancellation, and support ticket creation. These are trailing indicators that measure failure after friction has occurred.

3. **Which missing event limits the model most?**
   - **`driver_arrived_at_restaurant`**: Because driver arrival at the merchant is completely unrecorded, operations cannot decompose pre-pickup time into **travel time to restaurant** vs **kitchen wait dwell time**.

4. **What would you instrument next?**
   - Implement an automated geofenced / Bluetooth BLE arrival beacon when the driver enters the restaurant premises.
   - Integrate an explicit "Order Bagged & Ready" button on the merchant tablet.
   - Enforce mandatory `order_id` validation on customer support ticket submissions.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.